In [1]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

llm = init_chat_model("openai:gpt-5-nano")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {
    "configurable": {
        "thread_id": "1",
    },
}

In [2]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [3]:
@tool
def get_human_feedback(poem: str):
    """
    Asks the user for feedback on the poem.
    Use this before returning the final response.
    """
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback


llm_with_tools = llm.bind_tools(tools=[get_human_feedback])  # langchain api


def chatbot(state: State):
    # tool 사용을 강제
    response = llm_with_tools.invoke(
        f"""
        You are an expert in making poems.

        Use the `get_human_feedback` tool to get feedback on your poem.

        Only after you receive positive feedback you can return the final poem.

        if you receive it, return the final poem.

        ALWAYS ASK FOR FEEDBACK FIRST.

        Here is the conversation history:

        {state["messages"]}
        """
    )  # langchain api
    return {"messages": [response]}

In [4]:
tool_node = ToolNode(
    tools=[get_human_feedback],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn),
)

In [5]:
result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "Please make a poem about Python code. Simply"},
        ]
    },
    config=config,
)

In [6]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code. Simply
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_zHDg5WIDm9dpvu2w6M0ugVHo)
 Call ID: call_zHDg5WIDm9dpvu2w6M0ugVHo
  Args:
    poem: Python sings in clean, bright lines,
Indent breathes, and logic shines.
Loops weave through branches, true and neat,
Functions ferry thoughts in soft, neat feet.
Lists and dictionaries hold our art,
Modules guide the map and chart.
In this gentle code, a universe grows,
Where every tap of keys quietly knows.


In [7]:
snapshot = graph.get_state(config)

snapshot.interrupts

(Interrupt(value='Here is the poem, tell me what you think\nPython sings in clean, bright lines,\nIndent breathes, and logic shines.\nLoops weave through branches, true and neat,\nFunctions ferry thoughts in soft, neat feet.\nLists and dictionaries hold our art,\nModules guide the map and chart.\nIn this gentle code, a universe grows,\nWhere every tap of keys quietly knows.', id='31c00874a598aeffc6a5529f8f02fea3'),)

In [8]:
snapshot.next

('tools',)

In [9]:
from langgraph.types import Command

response = Command(resume="It looks great.")  # feedback


# resuming graph
result = graph.invoke(
    response,  # State or Command
    config=config,
)

In [10]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code. Simply
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_zHDg5WIDm9dpvu2w6M0ugVHo)
 Call ID: call_zHDg5WIDm9dpvu2w6M0ugVHo
  Args:
    poem: Python sings in clean, bright lines,
Indent breathes, and logic shines.
Loops weave through branches, true and neat,
Functions ferry thoughts in soft, neat feet.
Lists and dictionaries hold our art,
Modules guide the map and chart.
In this gentle code, a universe grows,
Where every tap of keys quietly knows.
================================= Tool Message =================================
Name: get_human_feedback

It looks great.
================================== Ai Message ==================================

Here is the final poem:

Python sings in clean, bright lines,
Indent breathes, and logic shines.
Loops weave through branches, true and neat,
Functions fe

In [11]:
snapshot = graph.get_state(config)

snapshot.next

()